In [1]:
import os
import polars as pl
from tqdm.notebook import tqdm

pl.Config(tbl_rows=50)

In [2]:
data_path = '../data/meds_outliers/'

In [3]:
data = pl.read_parquet('../data/meds_outliers/data/train/1.parquet')

In [4]:
# pl.Config(fmt_str_lengths=100000)
# data.with_columns(pl.col('code').str.split('//').list.len().alias('length')).group_by(['code_type','length']).first()['code_type','length','code']

In [5]:
data.filter(pl.col('code_type') == 'ICU-FLUID-OUTPUT').head(1)

subject_id,seq_id,out_id,er_id,hadm_id,icustay_id,disch_id,time,code,numeric_value,text_value,itemid,died_in_hosp,icu_los,admission_type,admission_location,discharge_location,diag_version,diag_icd_code,diag_seq_num,drg_severity,drg_mortality,drg_type,drg_code,priority,specimen_id,lab_lower_limit,lab_upper_limit,lab_flag,lab_unit,lab_itemid,gender,route,frequency,doses_per_24_hrs,medication,proc_seq_num,proc_version,proc_icd_code,micro_specimen_id,micro_org_name,micro_test_name,micro_spec_type_desc,micro_test_itemid,icu_care_unit,category,label,abbreviation,rate,unit,amount,amountuom,ordercategorydescription,ordercategoryname,secondaryordercategoryname,ordercomponenttypedescription,table,race,code_type,icd9_to_icd10_d,icd9_to_icd10_p,clean_medication,lab_label,lab_fluid,lab_category,lab_description,lab_frequency,time_diff,numeric_value/is_inlier
i64,f64,f64,f64,f64,f64,f64,datetime[μs],str,f64,str,f64,f64,f64,str,str,str,f64,str,f64,f64,f64,str,f64,str,f64,f64,f64,str,str,f64,str,str,str,f64,str,f64,f64,str,f64,str,str,str,f64,str,str,str,str,f64,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,bool
10052769,2.2087051e7,null,null,2.2087051e7,3.883265e7,null,2124-04-26 13:41:00,"""ICU-FLUID-OUTPUT//226560//Void""",400.0,null,226560.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Output""","""Void""","""Void""",null,null,null,null,null,null,null,null,"""icu/inputevents""",null,"""ICU-FLUID-OUTPUT""",null,null,"""UNK""",null,null,null,null,null,0.007639,true


In [6]:
descemb_mapping = {'MICROBIOLOGY': ['micro_test_name'],
'DIAGNOSIS-ICD': ['code_type','icd9_to_icd10_d'],
'MEDS_DEATH': ['code_type'],
'EMERGENCY-END': ['code_type'],
'EMERGENCY-START': ['code_type'],
'ADMISSION-AT-HOSPITAL': ['code_type'],
'MEDICATION': ['clean_medication','route','doses_per_24_hrs'],
'DISCHARGE-FROM-ICU': ['code_type'],
'OUTPATIENT-END': ['code_type'],
'ICU-CHART': ['label','numeric_value' 'unitname'],
'DISCHARGE-FROM-HOSPITAL': ['code_type'],
'ADMISSION-LOCATION': ['code_type'], #  and use .split('//')[-1] as second element of the list]
'LAB': ['lab_label', 'numeric_value', 'lab_unit'],
'RACE': ['code_type','race'],
'AGE_AT_ADMISSION': ['code_type','numeric_value'],
'PROCEDURE-ICD': ['code_type','icd9_to_icd10_p'],
'TIME-GAP': ['code_type','numeric_value'],
'DRG': ['code_type','drg_type','drg_code', 'drg_mortality'],
'ADMISSION-TYPE':['code_type'], #  and use .split('//')[-1] as second element of the list]
'ADMISSION-AT-ICU': ['code_type'],
'ICU-INFUSION': ['label','numeric_value', 'amountuom'],
'DISCHARGE-lOCATION': ['code_type','discharge_location'],
'GENDER': ['code_type','gender'],
'ICU-FLUID-OUTPUT': ['label','numeric_value' 'unitname'],
'OUTPATIENT-START': ['code_type'],
'ICU-PROCEDURE': ['label'],
}

In [7]:
import polars as pl

def dsva_number(x):
    """
    Digit-Split Value Aggregation with max 3 decimals:
    12.34567  -> '1 2 . 3 5'
    5         -> '5'
    5.1       -> '5 . 1'
    """
    if x is None:
        return None
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None

    # round to 3 decimal places
    v = round(v, 3)

    # format to 3 decimals, then strip trailing zeros and dot
    s = f"{v:.3f}".rstrip("0").rstrip(".")

    # DSVA: split into individual characters separated by spaces
    return " ".join(list(s))


def dsva_expr(col: pl.Expr) -> pl.Expr:
    return (
        col.cast(pl.Float64)
           .map_elements(dsva_number, return_dtype=pl.Utf8)
    )


In [8]:
gender_norm = (
    pl.when(pl.col("gender").str.to_lowercase() == "m")
      .then(pl.lit("male"))
    .when(pl.col("gender").str.to_lowercase() == "f")
      .then(pl.lit("female"))
    .otherwise(pl.col("gender"))
)


In [9]:
def normalize_code_type():
    return (
        pl.col("code_type")
        .str.replace_all(r"[_\-]+", " ")   # replace underscores and dashes with space
        .str.to_lowercase()
        .str.strip()
    )

In [10]:
def add_descemb(df: pl.DataFrame) -> pl.DataFrame:
    ct = pl.col("code_type")
    code = pl.col("code")

    # clean code type: remove - and _, lowercase
    code_type_clean = (
        ct.str.replace_all(r"[_\-]+", " ")
          .str.to_lowercase()
          .str.strip_chars()
    )

    # last element after the last "//"
    code_tail = code.str.extract(r"([^/]+)$").str.to_lowercase()

    descemb_expr = (
        pl.when(ct == "MICROBIOLOGY")
          .then(
              pl.col("micro_test_name").str.to_lowercase()
          )

        .when(ct == "DIAGNOSIS-ICD")
          .then(
              pl.concat_str(
                  [pl.lit("diagnosis icd 10 code"), pl.col("icd9_to_icd10_d")],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "MEDS_DEATH")
          .then(code_type_clean)

        .when(ct == "EMERGENCY-END")
          .then(code_type_clean)

        .when(ct == "EMERGENCY-START")
          .then(code_type_clean)

        .when(ct == "ADMISSION-AT-HOSPITAL")
          .then(code_type_clean)

        .when(ct == "MEDICATION")
          .then(
              pl.concat_str(
                  [
                      pl.col("clean_medication"),
                      pl.col("route"),
                      pl.col("doses_per_24_hrs")
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "DISCHARGE-FROM-ICU")
          .then(code_type_clean)

        .when(ct == "OUTPATIENT-END")
          .then(code_type_clean)

        .when(ct == "ICU-CHART")
          .then(
              pl.concat_str(
                  [
                      pl.col("label"),
                      dsva_expr(pl.col("numeric_value")),
                      pl.col("unitname")  # change if your col is named differently
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "DISCHARGE-FROM-HOSPITAL")
          .then(code_type_clean)

        .when(ct == "ADMISSION-LOCATION")
          .then(
              pl.concat_str(
                  [code_type_clean, code_tail],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "LAB")
          .then(
              pl.concat_str(
                  [
                      pl.col("lab_label"),
                      dsva_expr(pl.col("numeric_value")),
                      pl.col("lab_unit")
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "RACE")
          .then(
              pl.concat_str(
                  [code_type_clean, pl.col("race")],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "AGE_AT_ADMISSION")
          .then(
              pl.concat_str(
                  [
                      code_type_clean,
                      dsva_expr(pl.col("numeric_value"))
                  ],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "PROCEDURE-ICD")
          .then(
              pl.concat_str(
                  [pl.lit("procedure icd 10 code"), pl.col("icd9_to_icd10_p")],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )


        .when(ct == "TIME-GAP")
          .then(
              pl.concat_str(
                  [code_type_clean, dsva_expr(pl.col("numeric_value"))],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "DRG")
          .then(
              pl.concat_str(
                  [   code_type_clean,
                      pl.col("drg_type"),
                      pl.col("drg_code"),
                      pl.col("drg_mortality")
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "ADMISSION-TYPE")
          .then(
              pl.concat_str(
                  [code_type_clean, code_tail],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "ADMISSION-AT-ICU")
          .then(code_type_clean)

        .when(ct == "ICU-INFUSION")
          .then(
              pl.concat_str(
                  [
                      pl.col("label"),
                      dsva_expr(pl.col("numeric_value")),
                      pl.col("amountuom")
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "DISCHARGE-lOCATION")
          .then(
              pl.concat_str(
                  [code_type_clean, pl.col("discharge_location")],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "GENDER")
          .then(
              pl.concat_str(
                  [code_type_clean, gender_norm],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "ICU-FLUID-OUTPUT")
          .then(
              pl.concat_str(
                  [
                      pl.col("label"),
                      dsva_expr(pl.col("numeric_value")),
                      pl.col("unitname")  # change if needed
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "OUTPATIENT-START")
          .then(code_type_clean)

        .when(ct == "ICU-PROCEDURE")
          .then(
              pl.col("label").str.to_lowercase()
          )

        .otherwise(pl.lit(None))
        .alias("descemb")
    )



    return df.with_columns(descemb_expr)#['subject_id',
#                                         'seq_id',
#                                         'out_id',
#                                         'er_id',
#                                         'hadm_id',
#                                         'icustay_id',
#                                         'disch_id',
#                                         'time',
#                                         'code',
#                                         'code_type',
#                                         'descemb']


In [11]:
genhpf_mapping = {'MICROBIOLOGY': ['code_type','micro_test_name', 'micro_spec_type_desc'],
'DIAGNOSIS-ICD': ['code_type','icd9_to_icd10_d'],
'MEDS_DEATH': ['code_type'],
'EMERGENCY-END': ['code_type'],
'EMERGENCY-START': ['code_type'],
'ADMISSION-AT-HOSPITAL': ['code_type'],
'MEDICATION': ['code_type','clean_medication','route','frequency','doses_per_24_hrs'],
'DISCHARGE-FROM-ICU': ['code_type'],
'OUTPATIENT-END': ['code_type'],
'ICU-CHART': ['code_type','category','label','numeric_value', 'unitname'],
'DISCHARGE-FROM-HOSPITAL': ['code_type'],
'ADMISSION-LOCATION': ['code_type'], #  and use .split('//')[-1] as second element of the list]
'LAB': ['code_type','lab_label', 'priority' ,'numeric_value', 'lab_unit','lab_lower_limit', 'lab_upper_limit', 'lab_flag'],
'RACE': ['code_type','race'],
'AGE_AT_ADMISSION': ['code_type','numeric_value'],
'PROCEDURE-ICD': ['code_type','icd9_to_icd10_p'],
'TIME-GAP': ['code_type','numeric_value'],
'DRG': ['code_type','drg_type','drg_code', 'drg_mortality'],
'ADMISSION-TYPE':['code_type'], #  and use .split('//')[-1] as second element of the list]
'ADMISSION-AT-ICU': ['code_type'],
'ICU-INFUSION': ['code_type', 'category', 'label','numeric_value', 'amountuom'],
'DISCHARGE-lOCATION': ['code_type','discharge_location'],
'GENDER': ['code_type','gender'],
'ICU-FLUID-OUTPUT': ['code_type','category','label','numeric_value', 'unitname'],
'OUTPATIENT-START': ['code_type'],
'ICU-PROCEDURE': ['code_type','label'],
}

In [12]:
FEATURE_NAME_MAP = {
    "micro_test_name": "test name",
    "micro_spec_type_desc": "test description",
    "clean_medication": "medication name",
    "doses_per_24_hrs": "dose per 24 hours",
    "numeric_value": "value",
    "lab_label": "lab name",
    "lab_unit": "unit",
    "lab_flag": "flag",
    "amountuom": "unit",
    "unitname": "unit",
    "icd9_to_icd10_d": "value",
    "icd9_to_icd10_p": "value",
}

In [13]:
CODETYPE = (
    pl.col("code_type")
      .str.replace_all(r"[_\-]+", " ")
      .str.to_lowercase()
      .str.strip_chars()
)

In [14]:
def value_expr(col: pl.Expr, name: str) -> pl.Expr:
    if name in ["numeric_value"]:
        return dsva_expr(col)
    if name in ["lab_lower_limit", "lab_upper_limit"]:
        return dsva_expr(col)
    if name in ["icd9_to_icd10_d", "icd9_to_icd10_p"]:
        return col  # already textual ICD10 code
    return col  # textual columns

In [15]:
def feature_pair(feature_col: str) -> pl.Expr:
    """Return expression: '<feature name> <value>' """
    feat_name = FEATURE_NAME_MAP.get(feature_col, feature_col.replace("_", " ").replace("-", " "))
    feat_name = feat_name.lower()

    col_expr = pl.col(feature_col)
    col_expr = value_expr(col_expr, feature_col)

    return pl.concat_str([pl.lit(f"{feat_name}: "), col_expr], separator="", ignore_nulls=True)


In [16]:
def build_genhpf_desc(df: pl.DataFrame, mapping: dict) -> pl.DataFrame:
    ct_clean = CODETYPE
    code_tail = pl.col("code").str.extract(r"([^/]+)$").str.to_lowercase()

    # Build master expression
    expr = pl.lit("")  # will be overridden per code_type

    for code_type, cols in mapping.items():
        parts = []

        # always start with cleaned code type:
        parts.append(
                    pl.concat_str(
                        [pl.lit("event type: "), ct_clean],
                        separator="",
                        ignore_nulls=True
                    )
                )

        # add each feature pair
        for col in cols:
            if col == "code_type":
                continue
            parts.append(feature_pair(col))

        # build text for this code_type
        text_expr = (
            pl.concat_str(parts, separator=", ", ignore_nulls=True)
              .str.to_lowercase()
        )

        # add conditional branch
        expr = (
            pl.when(pl.col("code_type") == code_type)
              .then(text_expr)
              .otherwise(expr)
        )

    return df.with_columns(expr.alias("genhpf"))['subject_id',
                                        'seq_id',
                                        'out_id',
                                        'er_id',
                                        'hadm_id',
                                        'icustay_id',
                                        'disch_id',
                                        'code',
                                        'time',
                                        'time_diff',
                                        'descemb',
                                        'genhpf']


In [17]:
# import os
# import polars as pl
# data_path = '../data/meds_outliers/data/train/'
# out_path = '../data/descemb_genhpf/data/train/'
# d_item = pl.read_csv('../resources/mimic-mapping/d_items.csv',
#                      infer_schema_length=100000,
#                     columns=['itemid','unitname'])

# for shard in tqdm(os.listdir(data_path)):
#     data = pl.read_parquet(os.path.join(data_path,shard))
#     data = data.with_columns(pl.col("itemid").cast(pl.Int64))
#     data = data.join(d_item, on='itemid', how='left')
#     data = add_descemb(data)
#     data = build_genhpf_desc(data, genhpf_mapping)
#     data = data.with_columns(pl.col(['time_diff'])).fill_null(0.0)
#     data.write_parquet(os.path.join(out_path,shard))

In [18]:
limits = {
    'within24_query': {512:  ['w24_start_512',  'w24_end_512' ]},
    'within48_query': {512:  ['w48_start_512',  'w48_end_512' ]},
    'within_stay_query': {512:  ['wStay_start_512',  'wStay_end_512' ]},
    }

In [19]:
# data_idx = pl.read_parquet('../downstream_idx.parquet')
# subject_id = data_idx[0]['subject_id'][0]
# hadm_id = data_idx[0]['hadm_id'][0]
# icustay_id = data_idx[0]['icustay_id'][0]
# shard = data_idx[0]['shard'][0]
# file = pl.read_parquet(os.path.join('../data/descemb_genhpf/data/train/',shard))
# seq = file.filter(pl.col('subject_id') == subject_id)
# seq

In [20]:
# from datasets import Dataset, Features, Sequence, Value
# data_idx = pl.read_parquet('../downstream_idx.parquet')

# def gen():
#     for i in range(len(data_idx)):
#         subject_id = data_idx[i]['subject_id'][0]
#         icustay_id = data_idx[i]['icustay_id'][0]
#         shard = data_idx[i]['shard'][0]
        
#         w24_start = data_idx[i]['w24_start_512'][0]
#         w24_end = data_idx[i]['w24_end_512'][0]
        
#         w48_start = data_idx[i]['w48_start_512'][0]
#         w48_end = data_idx[i]['w48_end_512'][0]
        
#         wstay_start = data_idx[i]['wStay_start_512'][0]
#         wstay_end = data_idx[i]['wStay_end_512'][0]
        
#         file = pl.read_parquet(os.path.join('../data/descemb_genhpf/data/train/',shard))
#         seq = file.filter(pl.col('subject_id') == subject_id)
        
        

#         yield {
#             "subject_id": subject_id,
#             "icustay_id": icustay_id,

            
#             "within24_descemb": seq[w24_start:w24_end,:]['descemb'].to_list(),
#             "within48_descemb": seq[w48_start:w48_end,:]['descemb'].to_list(),
#             "within_stay_descemb": seq[wstay_start:wstay_end,:]['descemb'].to_list(),
            
#             "within24_genhpf": seq[w24_start:w24_end,:]['genhpf'].to_list(),
#             "within48_genhpf": seq[w48_start:w48_end,:]['genhpf'].to_list(),
#             "within_stay_genhpf": seq[wstay_start:wstay_end,:]['genhpf'].to_list(),
            
#             "within24_time": seq[w24_start:w24_end, :]["time"].to_list(),
#             "within48_time": seq[w48_start:w48_end, :]["time"].to_list(),
#             "within_stay_time": seq[wstay_start:wstay_end, :]["time"].to_list(),

#             "within24_time_diff": seq[w24_start:w24_end, :]["time_diff"].to_list(),
#             "within48_time_diff": seq[w48_start:w48_end, :]["time_diff"].to_list(),
#             "within_stay_time_diff": seq[wstay_start:wstay_end, :]["time_diff"].to_list(),
            
            
#             "within24_remed": seq[0:w24_end,:]['genhpf'].to_list(),
#             "within24_remed_time": seq[0:w24_end, :]["time"].to_list(),
#             "within24_remed_time_diff": seq[0:w24_end, :]["time_diff"].to_list(),
            
#             "within48_remed": seq[0:w48_end,:]['genhpf'].to_list(),
#             "within48_remed_time": seq[0:w48_end, :]["time"].to_list(),
#             "within48_remed_time_diff": seq[0:w48_end, :]["time_diff"].to_list(),
            
#             "within_stay_remed": seq[0:wstay_end,:]['genhpf'].to_list(),
#             "within_stay_remed_time": seq[0:wstay_end, :]["time"].to_list(),
#             "within_stay_remed_time_diff": seq[0:wstay_end, :]["time_diff"].to_list(),
            
#             }

# features = Features({
#     "subject_id": Value("int64"),
#     "icustay_id": Value("int64"),


#     "within24_descemb": Sequence(Value("string")),
#     "within48_descemb": Sequence(Value("string")),
#     "within_stay_descemb": Sequence(Value("string")),

#     "within24_genhpf": Sequence(Value("string")),
#     "within48_genhpf": Sequence(Value("string")),
#     "within_stay_genhpf": Sequence(Value("string")),


#     "within24_time": Sequence(Value("timestamp[us]")),
#     "within48_time": Sequence(Value("timestamp[us]")),
#     "within_stay_time": Sequence(Value("timestamp[us]")),

#     "within24_time_diff": Sequence(Value("float32")),
#     "within48_time_diff": Sequence(Value("float32")),
#     "within_stay_time_diff": Sequence(Value("float32")),

#     "within24_remed": Sequence(Value("string")),
#     "within48_remed": Sequence(Value("string")),
#     "within_stay_remed": Sequence(Value("string")),

#     "within24_remed_time": Sequence(Value("timestamp[us]")),
#     "within48_remed_time": Sequence(Value("timestamp[us]")),
#     "within_stay_remed_time": Sequence(Value("timestamp[us]")),

#     "within24_remed_time_diff": Sequence(Value("float32")),
#     "within48_remed_time_diff": Sequence(Value("float32")),
#     "within_stay_remed_time_diff": Sequence(Value("float32")),
# })

# ds_arrow = Dataset.from_generator(
#     gen,
#     features=features,
#     writer_batch_size=1000  # tune for shard sizes
# )

# # write Arrow shards to disk (memory-mappable)
# ds_arrow.save_to_disk("desc_gen_dataset")

# # optional: set PyTorch formatting
# # ds_arrow.set_format(type="torch")



# # # later / in training script:
# # from datasets import load_from_disk
# # train = load_from_disk("ehr_arrow_dataset")
# # train.set_format(type="torch")

In [21]:
from datasets import load_from_disk

In [22]:
# a = load_from_disk('./desc_gen_dataset/')

In [23]:
import os
import polars as pl
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk
from transformers import AutoTokenizer


class DescEmbDataset(Dataset):
    def __init__(
        self,
        dataset_path: str,
        data_idx_path: str,
        task: str = "y_mort",
        main_window: str = "within48_descemb",  
        split: str = "train",
        max_word_len: int = 32,                 
        max_events: int = None,                 
    ) -> None:

        self.task = task
        self.main_window = main_window
        self.max_word_len = max_word_len
        self.max_events = max_events
        

        self.data_idx = pl.scan_parquet(data_idx_path).collect()
        self.data_idx = self.data_idx.filter(pl.col("split") == split)
        self.data_idx = self.data_idx.filter(~pl.col("subject_id").is_in([15409850,16816440,18757959]) )
        
        

        self.hf_dataset = load_from_disk(dataset_path)

        subj_ids = self.hf_dataset["subject_id"]
        icu_ids = self.hf_dataset["icustay_id"]
        self._hf_index = {
            (int(s), int(i)): idx for idx, (s, i) in enumerate(zip(subj_ids, icu_ids))
        }


        self.tokenizer = AutoTokenizer.from_pretrained(
            "google/bert_uncased_L-2_H-128_A-2"
        )

    def __len__(self) -> int:
        return len(self.data_idx)

    def __getitem__(self, idx: int):
        row = self.data_idx.row(idx, named=True)
        subject_id = int(row["subject_id"])
        icustay_id = int(row["icustay_id"])
        label = row[self.task]

        ex = self.hf_dataset[idx]

        events = ex[self.main_window]  

        if self.max_events is not None and len(events) > self.max_events:
            events = events[: self.max_events]

        enc = self.tokenizer(
            events,
            padding="max_length",
            truncation=True,
            max_length=self.max_word_len,
            add_special_tokens=True,
            return_tensors="pt",
        )

        input_ids = enc["input_ids"]         
        attention_mask = enc["attention_mask"]

        seq_len = torch.tensor(len(events), dtype=torch.long)
        label = torch.tensor(label, dtype=torch.float)  

        return {
            "input_ids": input_ids,                 
            "attention_mask": attention_mask,                   
            "label": label                      
        }

In [24]:
# das = DescEmbDataset(dataset_path='./desc_gen_dataset/',
#                     data_idx_path='../downstream_idx.parquet',
#                     max_events=511)

In [25]:
# das.data_idx.filter(pl.col('subject_id').is_in([15409850,16816440,18757959]))

In [26]:
# das.data_idx[:5]

In [27]:
from torchmetrics.classification import BinaryAUROC, BinaryAveragePrecision

def get_bootstrap_ci(
    y_true: torch.Tensor,
    y_score: torch.Tensor,
    num_iter: int = 1000,
    alpha: float = 0.05,
    ndigits: int = 3,
):
    device = y_score.device

    y_true = y_true.detach().view(-1).to(device).long()
    y_score = y_score.detach().view(-1).to(device)

    auroc_point = BinaryAUROC().to(device)(y_score, y_true)
    auprc_point = BinaryAveragePrecision().to(device)(y_score, y_true)

    n = y_true.numel()
    auroc_samples = torch.empty(num_iter, device=device)
    auprc_samples = torch.empty(num_iter, device=device)

    for i in range(num_iter):
        idx = torch.randint(0, n, (n,), device=device)
        auroc_samples[i] = BinaryAUROC().to(device)(y_score[idx], y_true[idx])
        auprc_samples[i] = BinaryAveragePrecision().to(device)(y_score[idx], y_true[idx])

    # Percentile CI
    q_low = alpha / 2.0         # 2.5%
    q_high = 1.0 - alpha / 2.0  # 97.5%

    auroc_low = torch.quantile(auroc_samples, q_low)
    auroc_high = torch.quantile(auroc_samples, q_high)

    auprc_low = torch.quantile(auprc_samples, q_low)
    auprc_high = torch.quantile(auprc_samples, q_high)

    def _fmt(point, low, high):
        p = float(point.detach().cpu())
        l = float(low.detach().cpu())
        h = float(high.detach().cpu())
        return f"{round(p, ndigits)} ({round(l, ndigits)}, {round(h, ndigits)})"

    auroc_text = _fmt(auroc_point, auroc_low, auroc_high)
    auprc_text = _fmt(auprc_point, auprc_low, auprc_high)

    return auroc_text, auprc_text


def gather_1d_varlen_pl(module, x: torch.Tensor) -> torch.Tensor:
    x = x.detach().view(-1)

    if not getattr(module, "trainer", None) or module.trainer.world_size == 1:
        return x

    device = x.device
    local_len = torch.tensor([x.numel()], device=device, dtype=torch.long)

    all_lens = module.all_gather(local_len).view(-1) 
    max_len = int(all_lens.max().item())

    if x.numel() < max_len:
        pad = torch.zeros(max_len - x.numel(), device=device, dtype=x.dtype)
        x_pad = torch.cat([x, pad], dim=0)
    else:
        x_pad = x

    x_gather = module.all_gather(x_pad)

    chunks = []
    for r in range(x_gather.shape[0]):
        chunks.append(x_gather[r, : int(all_lens[r].item())])
    return torch.cat(chunks, dim=0)


def log_bootstrap_ci_text_percentile(
    module,
    y_true: torch.Tensor,
    y_score: torch.Tensor,
    prefix: str = "test",
    num_iter: int = 1000,
    alpha: float = 0.05,
    ndigits: int = 3,
):
    y_all = gather_1d_varlen_pl(module, y_true)
    s_all = gather_1d_varlen_pl(module, y_score)

    if not getattr(module, "trainer", None) or module.trainer.is_global_zero:
        auroc_ci_text, auprc_ci_text = get_bootstrap_ci(
            y_true=y_all,
            y_score=s_all,
            num_iter=num_iter,
            alpha=alpha,
            ndigits=ndigits,
        )
        try:
            module.log(f"{prefix}_auroc_ci", auroc_ci_text, logger=True, rank_zero_only=True)
            module.log(f"{prefix}_auprc_ci", auprc_ci_text, logger=True, rank_zero_only=True)
        except:
            print(f'AUROC= {auroc_ci_text}')
            print(f'AUPRC= {auprc_ci_text}')

In [28]:
# ds = DescEmbDataset(data_idx_path='../downstream_idx.parquet',
#                     dataset_path='./desc_gen_dataset/',
#                     main_window='within48_descemb',
#                     split='train',
#                     max_word_len=12,
#                     max_events=511)
# from collections import Counter
# ds = load_from_disk("./desc_gen_dataset")   # your path
# field = "within48_descemb"                  # change per window

# empty = []
# bad_types = Counter()

# for i in tqdm(range(len(ds))):
#     ev = ds[i][field]                       # list[str]
#     if ev is None:
#         empty.append(i); bad_types["None"] += 1
#         continue
#     if not isinstance(ev, list):
#         empty.append(i); bad_types[str(type(ev))] += 1
#         continue
#     # treat "", "   ", None as empty
#     ev2 = [e for e in ev if isinstance(e, str) and e.strip()]
#     if len(ev2) == 0:
#         empty.append(i); bad_types["empty_list_or_blank_strings"] += 1

# print("empty_count:", len(empty))
# print("example_indices:", empty[:20])
# print("breakdown:", bad_types)

In [29]:
# for i in empty[:20]:
#     print(i, ds[i]["subject_id"], ds[i]["icustay_id"])

In [30]:
import torch

class DescEmbCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, batch):


        batch = [b for b in batch if b["input_ids"] is not None]
        if len(batch) == 0:
            return {}

        lengths = [b["input_ids"].shape[0] for b in batch]
        max_S = max(lengths)
        W = batch[0]["input_ids"].shape[1]
        B = len(batch)

        input_ids = torch.full((B, max_S, W), self.pad_token_id, dtype=torch.long)
        attention_mask = torch.zeros((B, max_S, W), dtype=torch.long)
        seq_len = torch.tensor(lengths, dtype=torch.long)
        labels = torch.stack([b["label"] for b in batch])

        for i, b in enumerate(batch):
            S_i = b["input_ids"].shape[0]
            input_ids[i, :S_i] = b["input_ids"]
            attention_mask[i, :S_i] = b["attention_mask"]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "seq_len": seq_len,
            "label": labels,
        }

# collate_fn = DescEmbCollator(ds.tokenizer.pad_token_id)

In [31]:
# dl = DataLoader(dataset=ds, batch_size=32,collate_fn=collate_fn, shuffle=True)

In [32]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoConfig


class BertEventEncoder(nn.Module):
    def __init__(
        self,
        bert_model_name: str = "google/bert_uncased_L-2_H-128_A-2",
        pred_embed_dim: int = 128,
        init_bert_random: bool = False,
    ):
        super().__init__()

        if init_bert_random:
            config = AutoConfig.from_pretrained(bert_model_name)
            self.bert = AutoModel.from_config(config)
        else:
            self.bert = AutoModel.from_pretrained(bert_model_name)

        hidden_size = self.bert.config.hidden_size
        self.post_encode_proj = nn.Linear(hidden_size, pred_embed_dim)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        B, S, W = input_ids.shape

        flat_ids = input_ids.view(B * S, W)
        flat_mask = attention_mask.view(B * S, W)

        outputs = self.bert(
            input_ids=flat_ids,
            attention_mask=flat_mask,
        )
        cls_emb = outputs.last_hidden_state[:, 0, :] 

        event_emb = self.post_encode_proj(cls_emb)  
        event_emb = event_emb.view(B, S, -1)        
        return event_emb

In [33]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class GRUEventHead(nn.Module):

    def __init__(self, pred_embed_dim: int=128, 
                 pred_hidden_dim: int=256, 
                 max_event_len: int =511,
                 n_layers: int = 1, 
                 dropout: float = 0.3, 
                 task: str = "binary"):
        super().__init__()
        self.pred_embed_dim = pred_embed_dim
        self.pred_hidden_dim = pred_hidden_dim
        self.n_layers = n_layers
        self.max_event_len = max_event_len
        self.task = task

        self.model = nn.GRU(
            input_size=self.pred_embed_dim,
            hidden_size=self.pred_hidden_dim,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=False,
            num_layers=self.n_layers,
        )

        out_dim = 18 if task == "diagnosis" else 1
        self.final_proj = nn.Linear(self.pred_hidden_dim, out_dim)

    def pack_pad_seq(self, x: torch.Tensor, lengths: torch.Tensor):
        lengths = lengths.view(-1).cpu()
        lengths[lengths > self.max_event_len] = self.max_event_len

        packed = pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )
        output, _ = self.model(packed)
        output_seq, output_len = pad_packed_sequence(
            output, batch_first=True, padding_value=0.0
        )
        return output_seq, output_len

    def forward(self, x: torch.Tensor, seq_len: torch.Tensor) -> torch.Tensor:
       
        self.model.flatten_parameters()

        output_seq, _ = self.pack_pad_seq(x, seq_len) 
        i = range(x.size(0))
        last_hidden = output_seq[i, -1, :]             

        logits = self.final_proj(last_hidden)          
        if logits.shape[-1] == 1:
            logits = logits.squeeze(-1)                
        return logits

In [34]:
import lightning as lt
import torch
import torch.nn as nn
from torchmetrics.classification import BinaryAUROC, BinaryAveragePrecision


class DescEmbEvalModel(lt.LightningModule):
    def __init__(
        self,
        config,
        lr: float = 1e-4,
        wd: float = 0.001,
        max_epochs: int = 100,
        dropout: float = 0.3,
        freeze: bool = False,
    ):
        super().__init__()
        self.save_hyperparameters()

        bert_model_name = getattr(config, "bert_model_name", "google/bert_uncased_L-2_H-128_A-2")
        pred_embed_dim = getattr(config, "pred_embed_dim", 128)
        pred_hidden_dim = getattr(config, "pred_hidden_dim", 256)
        max_event_len = getattr(config, "max_event_len", 511)
        task = getattr(config, "task", "binary")

        self.encoder = BertEventEncoder(
            bert_model_name=bert_model_name,
            pred_embed_dim=pred_embed_dim,
            init_bert_random=getattr(config, "init_bert_random", False),
        )
        self.classifier = GRUEventHead(
            pred_embed_dim=pred_embed_dim,
            pred_hidden_dim=pred_hidden_dim,
            max_event_len=max_event_len,
            n_layers=getattr(config, "rnn_layer", 1),
            dropout=dropout,
            task=task,
        )

        if freeze:
            for p in self.encoder.parameters():
                p.requires_grad = False
            for p in self.classifier.parameters():
                p.requires_grad = True

        self.lr = lr
        self.wd = wd
        self.max_epochs = max_epochs

        self.criterion = nn.BCEWithLogitsLoss()

        self.train_step_preds = []
        self.train_step_label = []
        self.val_step_preds = []
        self.val_step_label = []
        self.test_step_preds = []
        self.test_step_label = []

        self.train_auroc = BinaryAUROC()
        self.train_auprc = BinaryAveragePrecision()
        self.val_auroc = BinaryAUROC()
        self.val_auprc = BinaryAveragePrecision()
        self.test_auroc = BinaryAUROC()
        self.test_auprc = BinaryAveragePrecision()

    def forward(self, input_ids, attention_mask, seq_len=None, labels=None):
        event_emb = self.encoder(input_ids=input_ids, attention_mask=attention_mask)  

        if seq_len is None:
            event_mask = attention_mask.any(dim=-1)     
            seq_len = event_mask.sum(dim=-1)             
        logits = self.classifier(event_emb, seq_len)       
        return logits

    def training_step(self, batch, batch_idx):
        logits = self.forward(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            seq_len=batch["seq_len"],
        )

        y = batch["label"].float().view(-1)
        loss = self.criterion(logits, y)

        pos_score = torch.sigmoid(logits)

        self.train_step_label.append(y)
        self.train_step_preds.append(pos_score)

        self.log("train_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_train_epoch_end(self) -> None:
        if len(self.train_step_label) == 0:
            return
        y = torch.cat(self.train_step_label)
        pos_score = torch.cat(self.train_step_preds)

        auroc = self.train_auroc(pos_score, y.long())
        auprc = self.train_auprc(pos_score, y.long())

        self.log("train_auroc", auroc, on_epoch=True, logger=True, prog_bar=True)
        self.log("train_auprc", auprc, on_epoch=True, logger=True, prog_bar=True)

        self.train_step_label.clear()
        self.train_step_preds.clear()

    def validation_step(self, batch, batch_idx):
        logits = self.forward(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            seq_len=batch["seq_len"],
        )

        y = batch["label"].float().view(-1)
        loss = self.criterion(logits, y)
        pos_score = torch.sigmoid(logits)

        self.val_step_label.append(y)
        self.val_step_preds.append(pos_score)

        self.log("val_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_validation_epoch_end(self, *args, **kwargs) -> None:
        if len(self.val_step_label) == 0:
            return
        y = torch.cat(self.val_step_label)
        pos_score = torch.cat(self.val_step_preds)

        auroc = self.val_auroc(pos_score, y.long())
        auprc = self.val_auprc(pos_score, y.long())

        self.log("val_auroc", auroc, on_epoch=True, logger=True, prog_bar=True)
        self.log("val_auprc", auprc, on_epoch=True, logger=True, prog_bar=True)

        self.val_step_label.clear()
        self.val_step_preds.clear()

    def test_step(self, batch, batch_idx):
        logits = self.forward(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            seq_len=batch["seq_len"],
        )

        y = batch["label"].float().view(-1)
        loss = self.criterion(logits, y)
        pos_score = torch.sigmoid(logits)

        self.test_step_label.append(y)
        self.test_step_preds.append(pos_score)

        self.log("test_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_test_epoch_end(self, *args, **kwargs) -> None:
        if len(self.test_step_label) == 0:
            return
        y = torch.cat(self.test_step_label)
        pos_score = torch.cat(self.test_step_preds)

        auroc = self.test_auroc(pos_score, y.long())
        auprc = self.test_auprc(pos_score, y.long())

        self.log("test_auroc", auroc, on_epoch=True, logger=True)
        self.log("test_auprc", auprc, on_epoch=True, logger=True)
        
        log_bootstrap_ci_text_percentile(
            module=self,
            y_true=y,
            y_score=pos_score,
            prefix="test",
            num_iter=1000,
            alpha=0.05,
            ndigits=3,
        )

        self.test_step_label.clear()
        self.test_step_preds.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.wd)
#         scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#             optimizer=optimizer,
#             eta_min=0,
#             T_max=self.max_epochs,
#         )
        return {"optimizer": optimizer}#, "lr_scheduler": scheduler}

In [35]:
from types import SimpleNamespace

config = SimpleNamespace(
    bert_model_name="google/bert_uncased_L-2_H-128_A-2",
    pred_embed_dim=128,
    pred_hidden_dim=128,     
    max_event_len=511,       
    rnn_layer=1,
    init_bert_random=False,  
    task="binary",           
)

In [36]:
# from torch.utils.data import DataLoader
# from transformers import AutoTokenizer


# dataset_path = "./desc_gen_dataset/"
# data_idx_path = "../downstream_idx.parquet"

# # datasets
# train_ds = DescEmbDataset(
#     dataset_path=dataset_path,
#     data_idx_path=data_idx_path,
#     task="y_mort",                
#     main_window="within48_descemb",
#     split="train",
#     max_word_len=32,
#     max_events=511,
# )

# val_ds = DescEmbDataset(
#     dataset_path=dataset_path,
#     data_idx_path=data_idx_path,
#     task="y_mort",
#     main_window="within48_descemb",
#     split="val",
#     max_word_len=32,
#     max_events=511,
# )

# # collator
# # tokenizer = AutoTokenizer.from_pretrained("google/bert_uncased_L-2_H-128_A-2")
# collator = DescEmbCollator(pad_token_id=train_ds.tokenizer.pad_token_id)

# # dataloaders
# train_dl = DataLoader(
#     train_ds,
#     batch_size=16,
#     shuffle=True,
#     num_workers=8,
#     collate_fn=collator,
# )

# val_dl = DataLoader(
#     val_ds,
#     batch_size=32,
#     shuffle=False,
#     num_workers=8,
#     collate_fn=collator,
# )

In [37]:
# batch = next(iter(train_dl))
# for k, v in batch.items():
#     print(k, v.shape if hasattr(v, "shape") else type(v))

# model = DescEmbEvalModel(
#     config=config,
#     lr=1e-4,
#     wd=0.00,
#     max_epochs=10,
#     dropout=0.3,
#     freeze=False,   # BERT-FT (full fine-tuning)
# )

# logits = model(
#     input_ids=batch["input_ids"],
#     attention_mask=batch["attention_mask"],
#     seq_len=batch["seq_len"],
# )
# print("logits shape:", logits)  # should be (B,)

In [38]:
# torch.set_float32_matmul_precision('high')
# trainer = lt.Trainer(
#     max_epochs=5,                              # one full pass over train_dl
#     accelerator="gpu" if torch.cuda.is_available() else "cpu",
#     devices=1,
#     log_every_n_steps=50,
#     precision='16-mixed'
    
    
# )

# trainer.fit(model,val_dl)

# GenHPF

In [39]:
import torch
from torch.utils.data import Dataset
from typing import Any, Dict, List, Optional, Tuple

import polars as pl
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer


class HierarchicalGenHPFDataset(Dataset):
    def __init__(
        self,
        dataset_path: str,
        data_idx_path: str,
        seq_field: str,
        label_field: Optional[str] = None,
        split: str = "train",
        tokenizer_name: str = "emilyalsentzer/Bio_ClinicalBERT",
        max_events: int = 511,
        max_tokens: int = 128,
    ) -> None:
        self.hf_dataset = self.hf_dataset = load_from_disk(dataset_path)
        self.seq_field = seq_field
        self.label_field = label_field
        self.max_events = max_events
        self.max_tokens = max_tokens

        df = pl.scan_parquet(data_idx_path).collect()
        self.data_idx = df.filter(pl.col("split") == split).to_pandas()

        subj = self.hf_dataset["subject_id"]
        stay = self.hf_dataset["icustay_id"]
        key_to_hf_idx: Dict[Tuple[int, int], int] = {}
        for i, (s, h) in enumerate(zip(subj, stay)):
            key_to_hf_idx[(int(s), int(h))] = i
        self.key_to_hf_idx = key_to_hf_idx


        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    def __len__(self) -> int:
        return len(self.data_idx)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.data_idx.iloc[idx]
        subject_id = int(row["subject_id"])
        icustay_id = int(row["icustay_id"])

        key = (subject_id, icustay_id)
        if key not in self.key_to_hf_idx:
            return {"input_ids": None, "label": None}

        hf_idx = self.key_to_hf_idx[key]
        hf_row = self.hf_dataset[hf_idx]

        events: List[str] = hf_row[self.seq_field]

        if len(events) > self.max_events:
            events = events[: self.max_events]


        if len(events) == 0:
            return {"input_ids": None, "label": None}

        enc = self.tokenizer(
            events,
            padding="max_length",
            truncation=True,
            max_length=self.max_tokens,
            add_special_tokens=True,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].long() 

        out: Dict[str, Any] = {
            "input_ids": input_ids,
        }

        if self.label_field is not None:
            y = row[self.label_field]
            out["label"] = torch.tensor(float(y), dtype=torch.float32)

        return out

In [40]:
import math
import torch
from torch import nn


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float, max_len: int):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)                      
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(1, max_len, d_model)                              
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)  

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)

In [41]:
from typing import List


class GenHPFEvalCollator:
    def __init__(self, pad_token_id: int = 0) -> None:
        self.pad_token_id = pad_token_id

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:

        batch = [b for b in batch if b["input_ids"] is not None]
        if len(batch) == 0:
            return {}

        input_ids_list = [b["input_ids"] for b in batch]  
        sizes = [x.size(0) for x in input_ids_list]
        B = len(input_ids_list)
        S_max = max(sizes)
        W = input_ids_list[0].size(1)


        collated_input_ids = input_ids_list[0].new_full(
            (B, S_max, W), fill_value=self.pad_token_id
        ).long()


        padding_mask = torch.ones(B, S_max, dtype=torch.bool)

        for i, (ids, S_i) in enumerate(zip(input_ids_list, sizes)):
            collated_input_ids[i, :S_i, :] = ids
            padding_mask[i, :S_i] = False

        out: Dict[str, Any] = {
            "input_ids": collated_input_ids,
            "padding_mask": padding_mask,
        }

        if "label" in batch[0] and batch[0]["label"] is not None:
            labels = torch.stack([b["label"] for b in batch])  # (B,)
            out["label"] = labels

        return out

In [42]:
import random
import polars as pl
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
from typing import Dict, Any, List, Tuple, Optional
from datasets import load_from_disk

class GenHPFSimCLRDataset(Dataset):
    def __init__(
        self,
        dataset_path: str,
        data_idx_path: str,
        split: str = "train",
        seq_field: str = "within_stay_remed",
        tokenizer_name: str = "emilyalsentzer/Bio_ClinicalBERT",
        max_events: int = 511,
        max_tokens: int = 128,
        min_events: int = 2,
        seed: int = 0,
    ):
        self.seq_field = seq_field
        self.max_events = max_events
        self.max_tokens = max_tokens
        self.min_events = min_events
        self.rng = random.Random(seed)


        df = pl.scan_parquet(data_idx_path).collect()
        df = df.filter(pl.col("split") == split)
        df = df.filter(~pl.col("subject_id").is_in([15409850, 16816440, 18757959]))
        self.data_idx = df.to_pandas()


        hf = load_from_disk(dataset_path)
        keep = [i for i, s in enumerate(hf["subject_id"]) if int(s) not in [15409850, 16816440, 18757959]]
        hf = hf.select(keep)
        self.hf_dataset = hf

        self.key_to_hf_idx: Dict[Tuple[int, int], int] = {
            (int(s), int(h)): i
            for i, (s, h) in enumerate(zip(hf["subject_id"], hf["icustay_id"]))
        }

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    def __len__(self) -> int:
        return len(self.data_idx)

    def _consecutive_slice(self, events: List[str]) -> List[str]:
        n = len(events)

        if n <= self.max_events:
            return events

        start = self.rng.randint(0, n - self.max_events)
        return events[start : start + self.max_events]

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.data_idx.iloc[idx]
        key = (int(row["subject_id"]), int(row["icustay_id"]))

        hf_idx = self.key_to_hf_idx.get(key, None)
        if hf_idx is None:
            return {"input_ids": None}

        events = self.hf_dataset[hf_idx].get(self.seq_field, None)
        if not events:
            return {"input_ids": None}

        events = [e for e in events if isinstance(e, str) and e.strip()]
        if len(events) < self.min_events:
            return {"input_ids": None}

        events = self._consecutive_slice(events)

        enc = self.tokenizer(
            events,
            padding="max_length",
            truncation=True,
            max_length=self.max_tokens,
            add_special_tokens=True,
            return_tensors="pt",
        )
        return {"input_ids": enc["input_ids"].long()}

In [43]:
class GenHPFSimCLRCollator:
    def __init__(
        self,
        pad_token_id: int,
        mask_token_id: int,
        cls_token_id: int,          
        mask_prob: float = 0.15,
    ) -> None:
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.cls_token_id = cls_token_id
        self.mask_prob = mask_prob

    def __call__(self, batch):
        batch = [b for b in batch if b["input_ids"] is not None]
        if len(batch) == 0:
            return {}

        v1s, v2s = [], []

        for b in batch:
            ev = b["input_ids"]  
            S = ev.size(0)
            if S <= 1:
                v1, v2 = ev, ev
            else:
                mid = S // 2
                v1, v2 = ev[:mid, :], ev[mid:, :]
            v1s.append(v1)
            v2s.append(v2)

        views = v1s + v2s  

        sizes = [v.size(0) for v in views]
        B2 = len(views)
        S_max = max(sizes)
        W = views[0].size(1)

        collated_input_ids = views[0].new_full(
            (B2, S_max, W), fill_value=self.pad_token_id
        ).long()

        padding_mask = torch.ones(B2, S_max, dtype=torch.bool) 

        for i, (v, S_i) in enumerate(zip(views, sizes)):
            collated_input_ids[i, :S_i, :] = v
            padding_mask[i, :S_i] = False

        b_idx, s_idx = torch.where(padding_mask)     # both are 1d, same length
        collated_input_ids[b_idx, s_idx, 0] = self.cls_token_id
        ids = collated_input_ids
        rand = torch.rand_like(ids, dtype=torch.float32)
        mask = (rand < self.mask_prob) & (ids != self.pad_token_id)
        ids[mask] = self.mask_token_id

        return {"input_ids": ids, "padding_mask": padding_mask}

In [44]:
# from datasets import load_from_disk

# # hf_ds = dx

# train_dataset = GenHPFSimCLRDataset(
#     dataset_path='./desc_gen_dataset/',
#     data_idx_path="../downstream_idx.parquet",
#     seq_field="within48_genhpf",
# #     label_field='y_mort',  
#     split="train",
#     tokenizer_name="emilyalsentzer/Bio_ClinicalBERT",
#     max_events=511,
#     max_tokens=128,
# )

# simclr_collator = GenHPFSimCLRCollator(pad_token_id=train_dataset.tokenizer.pad_token_id,
#                                        mask_token_id=train_dataset.tokenizer.mask_token_id,
#                                       cls_token_id=train_dataset.tokenizer.cls_token_id)


# eval_collator = GenHPFEvalCollator(train_dataset.tokenizer.pad_token_id)
# from torch.utils.data import DataLoader

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=16,
#     shuffle=True,
#     collate_fn=eval_collator,
# )

In [45]:
from typing import Tuple, Optional


class GenHPFEncoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        pad_token_id: int,
        encoder_embed_dim: int = 128,
        encoder_layers: int = 2,
        encoder_ffn_embed_dim: int = 512,
        encoder_attention_heads: int = 4,
        agg_embed_dim: int = 128,
        agg_layers: int = 4,
        agg_ffn_embed_dim: int = 512,
        agg_attention_heads: int = 4,
        dropout: float = 0.1,
        max_token_len: int = 128,
        max_events: int = 511,
        encoder_only: bool = False,
        ckpt_path: str = None
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.encoder_only = encoder_only
        self.pad_token_id = pad_token_id
        self.encoder_embed_dim = encoder_embed_dim
        self.agg_embed_dim = agg_embed_dim

        self.word_embeddings = nn.Embedding(vocab_size, 
                                            encoder_embed_dim, 
                                            padding_idx=pad_token_id)

        
        self.token_pos_encoding = PositionalEncoding(d_model=encoder_embed_dim, 
                                                     dropout=dropout, 
                                                     max_len=max_token_len)

        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=encoder_embed_dim,
            nhead=encoder_attention_heads,
            dim_feedforward=encoder_ffn_embed_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.event_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=encoder_layers
        )

        
        self.post_encode_proj = nn.Linear(encoder_embed_dim, agg_embed_dim)

        
        self.event_pos_encoding = PositionalEncoding(
            d_model=agg_embed_dim, dropout=dropout, max_len=max_events
        )

        agg_layer = nn.TransformerEncoderLayer(
            d_model=agg_embed_dim,
            nhead=agg_attention_heads,
            dim_feedforward=agg_ffn_embed_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.event_aggregator = nn.TransformerEncoder(
            agg_layer, num_layers=agg_layers
        )

        self.event_layer_norm = nn.LayerNorm(encoder_embed_dim, eps=1e-12)
        self.agg_layer_norm = nn.LayerNorm(agg_embed_dim, eps=1e-12)
        
        if ckpt_path:
            ckpt = torch.load(ckpt_path, map_location="cpu")
            state_dict = ckpt.get("state_dict", ckpt)
            
            cleaned = {}
            for k, v in state_dict.items():
                if k.startswith("model.encoder."):
                    cleaned[k.replace("model.encoder.", "")] = v
                elif k.startswith("encoder."):
                    cleaned[k.replace("encoder.", "")] = v
                else:
                    cleaned[k] = v

            missing, unexpected = self.load_state_dict(cleaned, strict=False)
            print(f"missing= \n{missing}")
            print(f"unexpected= \n{unexpected}")

    def forward(
        self,
        input_ids: torch.Tensor,         
        padding_mask: Optional[torch.Tensor] = None,  
#         encoder_only: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, S, W = input_ids.shape

        flat_ids = input_ids.view(B * S, W)


        x_tok = self.word_embeddings(flat_ids)
        x_tok = self.token_pos_encoding(x_tok)
        x_tok = self.event_layer_norm(x_tok)

        token_pad_mask = flat_ids.eq(self.pad_token_id) 


        x_tok = self.event_encoder(
            x_tok, src_key_padding_mask=token_pad_mask
        ) 

        if token_pad_mask.any():
            x_tok = x_tok.masked_fill(token_pad_mask.unsqueeze(-1), 0.0)
            lengths = (~token_pad_mask).sum(dim=1).clamp(min=1).unsqueeze(-1)
        else:
            lengths = torch.full(
                (B * S, 1), W, device=x_tok.device, dtype=torch.long
            )

        event_emb = x_tok.sum(dim=1) / lengths 

    
        event_emb = self.post_encode_proj(event_emb) 
        event_emb = event_emb.view(B, S, -1)        

        if padding_mask is None:
            padding_mask = input_ids.eq(self.pad_token_id).all(dim=2)  


        event_emb = self.event_pos_encoding(event_emb)
        event_emb = self.agg_layer_norm(event_emb)

        if self.encoder_only:
            return event_emb, padding_mask

        x = self.event_aggregator(
            event_emb, src_key_padding_mask=padding_mask
        ) 

        return x, padding_mask

In [46]:
import torch.nn as nn
import torch.nn.functional as F

class GenHPFSimCLRModel(nn.Module):
    def __init__(self, encoder: GenHPFEncoder, proj_dim: int = 128):
        super().__init__()
        self.encoder = encoder

        D = encoder.agg_embed_dim
        self.proj = nn.Sequential(
            nn.Linear(D, D),
            nn.ReLU(),
            nn.Linear(D, proj_dim),
        )

    def forward(self, input_ids: torch.Tensor, padding_mask: torch.Tensor) -> torch.Tensor:
        x, pad_mask = self.encoder(input_ids, padding_mask=padding_mask)

        if pad_mask is not None and pad_mask.any():
            x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)

        lengths = (~pad_mask).sum(dim=1).clamp(min=1).unsqueeze(-1)
        pooled = x.sum(dim=1) / lengths

        z = self.proj(pooled)
        return z

In [47]:
class GenHPFClassifier(nn.Module):

    def __init__(
        self,
        encoder: GenHPFEncoder,
        num_outputs: int = 1,  
    ):
        super().__init__()
        self.encoder = encoder
        self.num_outputs = num_outputs

        self.classifier = nn.Linear(encoder.agg_embed_dim, num_outputs)

    def forward(
        self,
        input_ids: torch.Tensor,
        padding_mask: torch.Tensor,
    ) -> torch.Tensor:

        x, pad_mask = self.encoder(input_ids, padding_mask=padding_mask)


        if pad_mask is not None and pad_mask.any():
            x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)


        lengths = (~pad_mask).sum(dim=1).clamp(min=1).unsqueeze(-1)
        pooled = x.sum(dim=1) / lengths

        logits = self.classifier(pooled)  
        if self.num_outputs == 1:
            logits = logits.squeeze(-1) 
        return logits

In [48]:
import lightning as lt
import torch
from torch import nn
from torch.optim import SGD
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchmetrics.classification import BinaryAUROC, BinaryAveragePrecision

In [49]:
class GenHPFDownstreamModule(lt.LightningModule):
    def __init__(
        self,
        encoder: GenHPFEncoder,
        num_outputs: int = 1,
        lr: float = 2e-5,
        wd: float = 1e-3,
        max_epochs: int = 100,
        pos_weight: float = 1.0,  
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["encoder"])

        self.model = GenHPFClassifier(
            encoder=encoder,
            num_outputs=num_outputs,
        )

        # loss
        if num_outputs == 1:
            self.criterion = nn.BCEWithLogitsLoss(
                pos_weight=torch.tensor(pos_weight)
            )
        else:
            self.criterion = nn.CrossEntropyLoss()

        if num_outputs == 1:
            self.train_auroc = BinaryAUROC()
            self.train_auprc = BinaryAveragePrecision()
            self.val_auroc = BinaryAUROC()
            self.val_auprc = BinaryAveragePrecision()
            self.test_auroc = BinaryAUROC()
            self.test_auprc = BinaryAveragePrecision()

        self.lr = lr
        self.wd = wd
        self.max_epochs = max_epochs

        self.train_step_preds = []
        self.train_step_label = []
        self.val_step_preds = []
        self.val_step_label = []
        self.test_step_preds = []
        self.test_step_label = []

    def forward(self, input_ids, padding_mask):
        logits = self.model(input_ids=input_ids, padding_mask=padding_mask)
        return logits


    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]       
        padding_mask = batch["padding_mask"] 
        y = batch["label"].float().view(-1) 

        logits = self.forward(input_ids, padding_mask) 

        if self.hparams.num_outputs == 1:
            loss = self.criterion(logits, y)
            pos_score = torch.sigmoid(logits)
            self.train_step_label.append(y.detach())
            self.train_step_preds.append(pos_score.detach())
        else:
            y_long = y.long()
            loss = self.criterion(logits, y_long)

        self.log("train_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_train_epoch_end(self):
        if self.hparams.num_outputs != 1:
            return

        y = torch.cat(self.train_step_label)
        pos_score = torch.cat(self.train_step_preds)

        auroc = self.train_auroc(pos_score, y.long())
        auprc = self.train_auprc(pos_score, y.long())

        self.log("train_auroc", auroc, on_epoch=True, logger=True, prog_bar=True)
        self.log("train_auprc", auprc, on_epoch=True, logger=True, prog_bar=True)

        self.train_step_label.clear()
        self.train_step_preds.clear()

    def validation_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        padding_mask = batch["padding_mask"]
        y = batch["label"].float().view(-1)

        logits = self.forward(input_ids, padding_mask)

        if self.hparams.num_outputs == 1:
            loss = self.criterion(logits, y)
            pos_score = torch.sigmoid(logits)
            self.val_step_label.append(y.detach())
            self.val_step_preds.append(pos_score.detach())
        else:
            y_long = y.long()
            loss = self.criterion(logits, y_long)

        self.log("val_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_validation_epoch_end(self):
        if self.hparams.num_outputs != 1:
            return

        y = torch.cat(self.val_step_label)
        pos_score = torch.cat(self.val_step_preds)

        auroc = self.val_auroc(pos_score, y.long())
        auprc = self.val_auprc(pos_score, y.long())

        self.log("val_auroc", auroc, on_epoch=True, logger=True, prog_bar=True)
        self.log("val_auprc", auprc, on_epoch=True, logger=True, prog_bar=True)

        self.val_step_label.clear()
        self.val_step_preds.clear()


    def test_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        padding_mask = batch["padding_mask"]
        y = batch["label"].float().view(-1)

        logits = self.forward(input_ids, padding_mask)

        if self.hparams.num_outputs == 1:
            loss = self.criterion(logits, y)
            pos_score = torch.sigmoid(logits)
            self.test_step_label.append(y.detach())
            self.test_step_preds.append(pos_score.detach())
        else:
            y_long = y.long()
            loss = self.criterion(logits, y_long)

        self.log("test_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_test_epoch_end(self):
        if self.hparams.num_outputs != 1:
            return

        y = torch.cat(self.test_step_label)
        pos_score = torch.cat(self.test_step_preds)

        auroc = self.test_auroc(pos_score, y.long())
        auprc = self.test_auprc(pos_score, y.long())

        self.log("test_auroc", auroc, on_epoch=True, logger=True)
        self.log("test_auprc", auprc, on_epoch=True, logger=True)
        
        log_bootstrap_ci_text_percentile(
            module=self,
            y_true=y,
            y_score=pos_score,
            prefix="test",
            num_iter=1000,
            alpha=0.05,
            ndigits=3,
        )

        self.test_step_label.clear()
        self.test_step_preds.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
#         scheduler = CosineAnnealingLR(
#             optimizer=optimizer,
#             eta_min=0.0,
#             T_max=self.max_epochs,
#         )
        return {"optimizer": optimizer}#, "lr_scheduler": scheduler}

In [50]:
import torch
import torch.nn.functional as F

class GenHPFSimCLRModule(lt.LightningModule):
    def __init__(
        self,
        encoder: GenHPFEncoder,
        lr: float = 1e-4,
        wd: float = 0.0,
        max_epochs: int = 100,
        temperature: float = 0.1,
        log_sim_every_n_steps: int = 20,  
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["encoder"])
        self.model = GenHPFSimCLRModel(encoder=encoder)
        self.lr = lr
        self.wd = wd
        self.max_epochs = max_epochs
        self.temperature = temperature
        self.log_sim_every_n_steps = log_sim_every_n_steps  

    def forward(self, input_ids, padding_mask):
        return self.model(input_ids=input_ids, padding_mask=padding_mask)

    def _nt_xent_loss(self, z: torch.Tensor) -> torch.Tensor:
        B2, D = z.shape
        assert B2 % 2 == 0
        B = B2 // 2

        z1 = F.normalize(z[:B], dim=1)
        z2 = F.normalize(z[B:], dim=1)
        reps = torch.cat([z1, z2], dim=0)                     

        sim = (reps @ reps.T) / self.temperature              
        diag = torch.eye(2 * B, device=sim.device, dtype=torch.bool)
        sim = sim.masked_fill(diag, float("-inf"))

        labels = torch.arange(2 * B, device=sim.device)
        labels = (labels + B) % (2 * B)

        return F.cross_entropy(sim, labels)

    @torch.no_grad()
    def log_simclr_cosines(self, z: torch.Tensor, prefix="simclr"):

        B2 = z.size(0)
        assert B2 % 2 == 0
        B = B2 // 2

        z = F.normalize(z, dim=1)
        z1, z2 = z[:B], z[B:]

        
        pos = F.cosine_similarity(z1, z2, dim=1) 
        pos_mean = pos.mean()

       
        reps = torch.cat([z1, z2], dim=0)                
        sim = reps @ reps.T                               

        diag = torch.eye(2*B, device=z.device, dtype=torch.bool)
        pos_mask = diag.roll(shifts=B, dims=1)            
        neg_mask = ~(diag | pos_mask)

        neg_mean = sim[neg_mask].mean()

        self.log(f"{prefix}_pos_cos", pos_mean, prog_bar=True, on_step=True, logger=True)
        self.log(f"{prefix}_neg_cos", neg_mean, prog_bar=True, on_step=True, logger=True)

    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        padding_mask = batch["padding_mask"]

        z = self.forward(input_ids, padding_mask)
        self.log_simclr_cosines(z)
        
        with torch.no_grad():
            z0 = F.normalize(z, dim=1)
            sim = z0 @ z0.T                     
            sim.fill_diagonal_(float("nan"))    

            mask = ~torch.isnan(sim)
            sim_vals = sim[mask]                 

            self.log("sim_mean", sim_vals.mean(), on_step=True, prog_bar=True)
            self.log("sim_std",  sim_vals.std(unbiased=False), on_step=True, prog_bar=True)
        loss = self._nt_xent_loss(z)


        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True, logger=True)


        return loss

    def configure_optimizers(self):
        return {"optimizer": torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.wd)}

In [51]:
import torch
from torch.utils.data import DataLoader
import lightning as lt
from datasets import load_from_disk
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")


encoder = GenHPFEncoder(
    vocab_size=tokenizer.vocab_size,
    pad_token_id=tokenizer.pad_token_id,
    encoder_embed_dim=128,
    encoder_layers=2,
    encoder_ffn_embed_dim=512,
    encoder_attention_heads=4,
    agg_embed_dim=128,
    agg_layers=4,
    agg_ffn_embed_dim=512,
    agg_attention_heads=4,
    dropout=0.3,
    max_token_len=64,   
    max_events=511,
    encoder_only=False,
    ckpt_path='/scratch/sas10092/ehr-foundation/models/simclr/wandb/run-20251224_094659-genhpf_simclr_genhpf_13612113_511_0_15_maskprob_12_5overlap/files/ckpt/epoch=62-step=21042.ckpt')





missing= 
[]
unexpected= 
['model.proj.0.weight', 'model.proj.0.bias', 'model.proj.2.weight', 'model.proj.2.bias']


In [52]:

# train_dataset = GenHPFSimCLRDataset(
#     dataset_path='./desc_gen_dataset/',
#     data_idx_path="../downstream_idx.parquet",
# #     seq_field="within48_genhpf",
# #     label_field='y_mort',  
#     split="train",
#     tokenizer_name="emilyalsentzer/Bio_ClinicalBERT",
#     max_events=511,
#     max_tokens=64,
# )

# simclr_collator = GenHPFSimCLRCollator(pad_token_id=train_dataset.tokenizer.pad_token_id,
#                                        mask_token_id=train_dataset.tokenizer.mask_token_id,
#                                       cls_token_id=train_dataset.tokenizer.cls_token_id)

# simclr_loader = DataLoader(
#     train_dataset,
#     batch_size=32,          
#     shuffle=True,
#     num_workers=8,
#     collate_fn=simclr_collator,
#     persistent_workers=True,
#     prefetch_factor=4
# )


# simclr_module = GenHPFSimCLRModule(
#     encoder=encoder,
#     lr=1e-4,
#     wd=0,
#     max_epochs=2,
#     temperature=0.1,
# )

# torch.set_float32_matmul_precision('high')
# trainer_simclr = lt.Trainer(
#     max_epochs=5,
# #     limit_train_batches=5,   
#     accelerator="auto",
#     devices=1,
#     log_every_n_steps=1,
#     precision='16-mixed'
# )

# trainer_simclr.fit(simclr_module, train_dataloaders=simclr_loader)

In [53]:
# data_idx_path = "../downstream_idx.parquet"              

# train_dataset_genhpf = HierarchicalGenHPFDataset(
#     dataset_path='./desc_gen_dataset/',
#     data_idx_path=data_idx_path,
#     seq_field="within48_genhpf",  
#     label_field="y_mort",         
#     split="train",
#     tokenizer_name="emilyalsentzer/Bio_ClinicalBERT",
#     max_events=511,
#     max_tokens=64,
# )

# val_dataset_genhpf = HierarchicalGenHPFDataset(
#     dataset_path='./desc_gen_dataset/',
#     data_idx_path=data_idx_path,
#     seq_field="within48_genhpf",
#     label_field="y_mort",
#     split="val",
#     tokenizer_name="emilyalsentzer/Bio_ClinicalBERT",
#     max_events=511,
#     max_tokens=64,
# )




# eval_collator = GenHPFEvalCollator(pad_token_id=tokenizer.pad_token_id)

# train_loader = DataLoader(
#     train_dataset_genhpf,
#     batch_size=16,
#     shuffle=True,
#     num_workers=8,
#     collate_fn=eval_collator,
# )

# val_loader = DataLoader(
#     val_dataset_genhpf,
#     batch_size=16,
#     shuffle=False,
#     num_workers=8,
#     collate_fn=eval_collator,
# )


# downstream_module = GenHPFDownstreamModule(
#     encoder=encoder,      
#     num_outputs=1,         
#     lr=2e-4,
#     wd=1e-3,
#     max_epochs=5,
#     pos_weight=1.0,        
# )


# trainer_downstream = lt.Trainer(
#     max_epochs=1,
# #     limit_train_batches=5,
#     limit_val_batches=2,
#     accelerator="auto",
#     devices=1,
#     log_every_n_steps=1,
#     precision='16-mixed'
# )

# trainer_downstream.fit(
#     downstream_module,
#     train_dataloaders=val_loader,
#     val_dataloaders=train_loader,
# )

In [54]:
# import polars as pl

In [55]:
# pl.read_parquet('../data/descemb_genhpf/data/train/0.parquet').head()

# REMed

In [56]:
from typing import Any, Dict, List, Optional, Tuple
import numpy as np
import polars as pl
import torch
from torch.utils.data import Dataset
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer


class REMedGenHPFPoolDataset(Dataset):

    def __init__(
        self,
        hf_path: str,                 
        data_idx_path: str,
        seq_field: str,               
        time_field: str,              
        time_diff_field: str,         
        label_field: Optional[str] = None,
        split: str = "train",
        tokenizer_name: str = "emilyalsentzer/Bio_ClinicalBERT",
        seq_len: int = 512,
        max_tokens: int = 128,
        random_sample_train: bool = True,
        deterministic_eval: bool = True,
        seed: int = 2020,
    ):
        full_ds = load_from_disk(hf_path)


        self.hf_dataset = full_ds.select_columns([
            "subject_id",
            "icustay_id",
            seq_field,
            time_field,
            time_diff_field,
        ])
        self.seq_field = seq_field
        self.time_field = time_field
        self.time_diff_field = time_diff_field
        self.label_field = label_field
        self.seq_len = int(seq_len)
        self.max_tokens = int(max_tokens)
        self.random_sample_train = random_sample_train
        self.deterministic_eval = deterministic_eval
        self.split = split


        subj = self.hf_dataset["subject_id"]
        stay = self.hf_dataset["icustay_id"]
        self.key_to_hf_idx: Dict[Tuple[int, int], int] = {
            (int(s), int(h)): i for i, (s, h) in enumerate(zip(subj, stay))
        }

        df = pl.scan_parquet(data_idx_path).collect()
        self.data_idx = df.filter(pl.col("split") == split).to_pandas()

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)


        self.base_seed = int(seed)

    def __len__(self) -> int:
        return len(self.data_idx)

    def _rng_for(self, subject_id: int, icustay_id: int) -> np.random.Generator:
        
        mix = (self.base_seed * 1_000_003) ^ (subject_id * 1009) ^ (icustay_id * 9176)
        mix = mix % (2**32 - 1)
        return np.random.default_rng(mix)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.data_idx.iloc[idx]
        subject_id = int(row["subject_id"])
        icustay_id = int(row["icustay_id"])

        key = (subject_id, icustay_id)
        if key not in self.key_to_hf_idx:
            return {"input_ids": None, "label": None}

        hf_row = self.hf_dataset[self.key_to_hf_idx[key]]

        events: List[str] = list(hf_row[self.seq_field])
        times_ts = list(hf_row[self.time_field])
        time_diff = list(hf_row[self.time_diff_field])


        L = len(events)
        if L == 0 or L != len(times_ts) or L != len(time_diff):
            return {"input_ids": None, "label": None}


        if self.split == "train" and self.random_sample_train:
            rng = self._rng_for(subject_id, icustay_id)
            if L >= self.seq_len:

                idxs = rng.choice(L, size=self.seq_len, replace=False)
            else:

                idxs = rng.choice(L, size=self.seq_len, replace=True)

            idxs = np.sort(idxs)
        else:
            # eval: deterministic
            if self.deterministic_eval:
                idxs = np.arange(max(0, L - self.seq_len), L)
            else:
                rng = self._rng_for(subject_id, icustay_id)
                if L >= self.seq_len:
                    idxs = np.sort(rng.choice(L, size=self.seq_len, replace=False))
                else:
                    idxs = np.sort(rng.choice(L, size=self.seq_len, replace=True))

        events_sel = [events[i] for i in idxs]
        times_sel = [times_ts[i] for i in idxs]
        td_sel = np.asarray([time_diff[i] for i in idxs], dtype=np.float32)

        enc = self.tokenizer(
            events_sel,
            padding="max_length",
            truncation=True,
            max_length=self.max_tokens,
            add_special_tokens=True,
            return_tensors="pt",
        )

        out: Dict[str, Any] = {
            "input_ids": enc["input_ids"].long(),    
            "times_ts": times_sel,                    
            "time_diff": torch.from_numpy(td_sel),    
            "subject_id": torch.tensor(subject_id, dtype=torch.int64),
            "icustay_id": torch.tensor(icustay_id, dtype=torch.int64),
        }

        if self.label_field is not None:
            y = row[self.label_field]
            out["label"] = torch.tensor(float(y), dtype=torch.float32)

        return out

In [57]:
from typing import Any, Dict, List, Optional
import torch

class REMedGenHPFCollator:
    def __init__(
        self,
        pad_token_id: int = 0,
        time_mode: str = "timestamp",   
        time_diff_unit: str = "days",   
    ) -> None:
        self.pad_token_id = pad_token_id
        assert time_mode in {"timestamp", "time_diff"}
        self.time_mode = time_mode
        assert time_diff_unit in {"days", "minutes"}
        self.time_diff_unit = time_diff_unit

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        batch = [b for b in batch if b.get("input_ids") is not None]
        if len(batch) == 0:
            return {}
        input_ids_list = [b["input_ids"] for b in batch]
        sizes = [x.size(0) for x in input_ids_list]
        B = len(input_ids_list)
        S_max = max(sizes)
        W = input_ids_list[0].size(1)

        collated_input_ids = input_ids_list[0].new_full(
            (B, S_max, W), fill_value=self.pad_token_id
        ).long()

        padding_mask = torch.ones(B, S_max, dtype=torch.bool)
        collated_times = torch.zeros(B, S_max, dtype=torch.float32)

        for i, b in enumerate(batch):
            ids = b["input_ids"]
            S_i = ids.size(0)

            collated_input_ids[i, :S_i, :] = ids
            padding_mask[i, :S_i] = False

            if self.time_mode == "timestamp":
                ts_list = b["times_ts"]
                anchor = None
                for t in ts_list:
                    if t is not None:
                        anchor = t
                        break
                if anchor is None:
                    
                    times_min = torch.zeros(S_i, dtype=torch.float32)
                else:
                    
                    vals = []
                    for t in ts_list:
                        if t is None:
                            vals.append(0.0)
                        else:
                            vals.append((t - anchor).total_seconds() / 60.0)
                    times_min = torch.tensor(vals, dtype=torch.float32)

            else:
                
                td = b["time_diff"].float()
                if self.time_diff_unit == "days":
                    td = td * 1440.0
                
                times_min = torch.cumsum(td, dim=0)
               
                if S_i > 0:
                    times_min = times_min - times_min[0]

            collated_times[i, :S_i] = times_min

        out: Dict[str, Any] = {
            "input_ids": collated_input_ids,
            "padding_mask": padding_mask,
            "times": collated_times,  
        }

        if "label" in batch[0] and batch[0]["label"] is not None:
            out["label"] = torch.stack([b["label"] for b in batch])

        return out

In [58]:
import math
import torch
import torch.nn as nn
from transformers.models.roformer.modeling_roformer import RoFormerConfig, RoFormerEncoder



class Retriever(nn.Module):
    def __init__(self, pred_dim: int):
        super().__init__()
        in_dim = pred_dim + 1  # repr + time
        self.model = nn.Sequential(
            nn.Linear(in_dim, in_dim // 2),
            nn.LayerNorm(in_dim // 2),
            nn.ReLU(),
            nn.Linear(in_dim // 2, in_dim // 4),
            nn.LayerNorm(in_dim // 4),
            nn.ReLU(),
            nn.Linear(in_dim // 4, in_dim // 8),
            nn.LayerNorm(in_dim // 8),
            nn.ReLU(),
            nn.Linear(in_dim // 8, 1),
            nn.Sigmoid(),
        )

    def forward(self, reprs: torch.Tensor, times: torch.Tensor) -> torch.Tensor:
        times = times.unsqueeze(-1).type(reprs.dtype)
        return self.model(torch.cat([reprs, times], dim=-1)).squeeze(-1)  # (B, L)


class ReprTimeEnc(nn.Module):
    def __init__(self, pred_dim: int, dropout: float, pred_time: int):
        super().__init__()
        self.pred_time = pred_time
        div_term = torch.exp(torch.arange(0, pred_dim, 2) * (-math.log(10000.0) / pred_dim))
        self.register_buffer("div_term", div_term)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x: torch.Tensor, times: torch.Tensor):

        times = self.pred_time * 60 - times
        src_pad_mask = x.eq(0).all(dim=-1)  # (B, L)

        pe = torch.zeros_like(x)
        pe[:, :, 0::2] = torch.sin(times.unsqueeze(-1) * self.div_term)
        pe[:, :, 1::2] = torch.cos(times.unsqueeze(-1) * self.div_term)
        x = self.dropout(x + pe)
        return x, src_pad_mask


class Predictor(nn.Module):
    def __init__(self, 
                 pred_dim: int=512, 
                 dropout: float=0.2, 
                 pred_time: int=48, 
                 n_layers: int=2, 
                 n_heads: int=8, 
                 max_len: int=128):
        super().__init__()
        self.time_enc = ReprTimeEnc(pred_dim, dropout, pred_time)
        config = RoFormerConfig(
            hidden_size=pred_dim,
            num_hidden_layers=n_layers,
            num_attention_heads=n_heads,
            intermediate_size=pred_dim * 4,
            hidden_dropout_prob=dropout,
            attention_probs_dropout_prob=dropout,
            max_position_embeddings=max_len,
        )
        self.model = RoFormerEncoder(config)

    def forward(self, reprs: torch.Tensor, times: torch.Tensor) -> torch.Tensor:
        x, src_pad_mask = self.time_enc(reprs, times)

        mask = src_pad_mask * torch.tensor(torch.finfo(x.dtype).min, dtype=x.dtype, device=x.device)
        mask = mask.unsqueeze(-1).unsqueeze(1)  
        return self.model(x, attention_mask=mask)["last_hidden_state"] 


class PredOutPutLayer(nn.Module):
    def __init__(self, pred_dim: int, num_classes: int = 1):
        super().__init__()
        self.num_classes = num_classes
        self.final_proj = nn.Linear(pred_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mask = x.ne(0).any(dim=-1).unsqueeze(-1)  
        pooled = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        logits = self.final_proj(pooled)  
        return logits



class REMed(nn.Module):
    def __init__(
        self,
        pred_dim: int = 512,
        n_heads: int = 8,
        n_layers: int = 2,
        dropout: float = 0.2,
        max_retrieve_len: int = 128,
        pred_time: int = 48,
        num_classes: int = 1,
    ):
        super().__init__()
        self.pred_dim = pred_dim
        self.max_retrieve_len = max_retrieve_len

        self.predictor = Predictor(pred_dim, dropout, pred_time, n_layers, n_heads, max_retrieve_len)
        self.emb2out_model = PredOutPutLayer(pred_dim, num_classes=num_classes)
        self.retriever = Retriever(pred_dim)

        self.register_buffer("random_token_emb", torch.randn(pred_dim))
        self.set_mode("retriever")

    def set_mode(self, mode: str):
        self.mode = mode
        if mode == "retriever":
            self.requires_grad_(False)
            self.retriever.requires_grad_(True)
        elif mode == "predictor":
            self.requires_grad_(True)
            self.retriever.requires_grad_(False)

    def forward(self, reprs: torch.Tensor, times: torch.Tensor) -> torch.Tensor:

        reprs = nn.functional.pad(reprs, (0, 0, 0, self.max_retrieve_len))
        times = nn.functional.pad(times, (0, self.max_retrieve_len))


        times = torch.where(reprs.eq(0).all(dim=-1), torch.tensor(1e10, device=times.device, dtype=times.dtype), times)

        sim = self.retriever(reprs, times)  # (B, L+Kpad)

        _sim = torch.where(reprs.eq(0).all(dim=-1), torch.zeros_like(sim), sim)
        topk_values, topk_indices = torch.topk(_sim, self.max_retrieve_len, dim=1)

        topk = torch.gather(reprs, 1, topk_indices.unsqueeze(-1).repeat(1, 1, self.pred_dim))
        topk_times = torch.gather(times, 1, topk_indices)
        B, K, E = topk.shape


        topk_times, order = topk_times.sort(dim=1)
        topk = topk.gather(1, order.unsqueeze(-1).repeat(1, 1, E))
        topk_values = topk_values.gather(1, order)

        def _retriever_path():
            _topk_values = topk_values.reshape(B * K, 1)
            _topk = topk.reshape(B * K, 1, -1)
            _topk_times = topk_times.reshape(B * K, 1)

            zero_idcs = _topk.eq(0).all(dim=-1)
            _topk_times = torch.where(zero_idcs, torch.zeros_like(_topk_times), _topk_times)
            _topk_values = torch.where(zero_idcs, torch.zeros_like(_topk_values), _topk_values)


            _topk = torch.where(zero_idcs.unsqueeze(-1), self.random_token_emb.expand(B * K, 1, E), _topk)
            _topk_values = _topk_values + 1e-10
            _topk_values = (_topk_values.reshape(B, K) / _topk_values.reshape(B, K).sum(dim=1, keepdim=True)).reshape(B * K)

            enc = self.predictor(_topk, _topk_times)
            logits = self.emb2out_model(enc) 


            logits = torch.sum((_topk_values.unsqueeze(-1) * logits).reshape(B, K, -1), dim=1)
            return logits

        def _predictor_path():
            # ensure first token isn't all-zeros
            topk[:, 0, :] = torch.where(
                topk[:, 0, :].sum(dim=-1, keepdim=True) == 0,
                self.random_token_emb.expand(B, E),
                topk[:, 0, :],
            )
            enc = self.predictor(topk, topk_times)
            logits = self.emb2out_model(enc) 
            return logits

        if self.training:
            if self.training and self.mode == "retriever":
                logits = _retriever_path()
            else:
                logits = _predictor_path()
        else:
            logits = _predictor_path()

        return logits



class REMedWithGenHPF(nn.Module):
    def __init__(
        self,
        genhpf_encoder: nn.Module,
        pred_dim: int = 512,
        num_classes: int = 1,
        pred_time: int = 48,
        max_retrieve_len: int = 128,
        n_heads: int = 8,
        n_layers: int = 2,
        dropout: float = 0.2,
        freeze_encoder: bool = True,
    ):
        super().__init__()
        self.encoder = genhpf_encoder
        self.freeze_encoder = freeze_encoder
        self.num_classes = num_classes

        # encoder output dim -> pred_dim (fixes agg_embed_dim=128 vs pred_dim=512 mismatch)
        enc_out_dim = getattr(genhpf_encoder, "agg_embed_dim", None)
        if enc_out_dim is None:
            raise ValueError("genhpf_encoder must expose agg_embed_dim (encoder output dim).")

        self.enc_to_pred = nn.Identity() if enc_out_dim == pred_dim else nn.Linear(enc_out_dim, pred_dim)

        self.remed = REMed(
            pred_dim=pred_dim,
            n_heads=n_heads,
            n_layers=n_layers,
            dropout=dropout,
            max_retrieve_len=max_retrieve_len,
            pred_time=pred_time,
            num_classes=num_classes,
        )

        if freeze_encoder:
            self.encoder.requires_grad_(False)
            self.encoder.eval()  # no dropout in encoder

    def forward(
        self,
        input_ids: torch.Tensor,
        padding_mask: torch.Tensor,
        times: torch.Tensor,
        return_probs: bool = False,
    ):
        # keep encoder deterministic if frozen (Lightning may flip modules to train())
        if self.freeze_encoder:
            self.encoder.eval()

        reprs, pad_mask = self.encoder(input_ids, padding_mask=padding_mask)  # (B,S,enc_out_dim)
        reprs = self.enc_to_pred(reprs)  # (B,S,pred_dim)

        if pad_mask is not None and pad_mask.any():
            reprs = reprs.masked_fill(pad_mask.unsqueeze(-1), 0.0)

        logits = self.remed(reprs, times.float())  # (B,C)

        if self.num_classes == 1:
            logits = logits.squeeze(-1)  # (B,)
            return (logits, torch.sigmoid(logits)) if return_probs else logits

        return (logits, torch.softmax(logits, dim=-1)) if return_probs else logits

In [59]:
import lightning as lt
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import LinearLR
from torchmetrics.classification import BinaryAUROC, BinaryAveragePrecision


class REMedLightningModule(lt.LightningModule):

    def __init__(
        self,
        model,  # REMedWithGenHPF, exposes .remed.set_mode(...)
        lr: float = 1e-5,
        max_epochs: int = 100,
        pos_weight: float = 1.0,
        freeze_encoder: bool = True,
        use_warmup: bool = False,
        warmup_steps: int = 500,
        num_classes: int = 1,   # <-- ADD BACK
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model = model
        self.automatic_optimization = False

        self.num_classes = num_classes

        if num_classes == 1:
            self.criterion = nn.BCEWithLogitsLoss(
                pos_weight=torch.tensor(pos_weight, dtype=torch.float32)
            )
            self.train_auroc = BinaryAUROC()
            self.train_auprc = BinaryAveragePrecision()
            self.val_auroc = BinaryAUROC()
            self.val_auprc = BinaryAveragePrecision()
            self.test_auroc = BinaryAUROC()
            self.test_auprc = BinaryAveragePrecision()
        else:
            self.criterion = nn.CrossEntropyLoss()
            # keep metrics out for multiclass unless you explicitly want them

        self.train_step_preds, self.train_step_label = [], []
        self.val_step_preds, self.val_step_label = [], []
        self.test_step_preds, self.test_step_label = [], []

        self._freeze_encoder = freeze_encoder

        # IMPORTANT: don't leave the model stuck in "predictor" mode from debug prints
        self.model.remed.set_mode("retriever")

    def on_train_batch_start(self, batch, batch_idx):
        if self._freeze_encoder and hasattr(self.model, "encoder"):
            self.model.encoder.eval()

    def forward(self, batch):
        return self.model(
            input_ids=batch["input_ids"],
            padding_mask=batch["padding_mask"],
            times=batch["times"],
        )

    def _step_once(self, batch, y, mode: str):
        opt = self.optimizers()
        sch = self.lr_schedulers()

        self.model.remed.set_mode(mode)

        logits = self.forward(batch)

        if self.num_classes == 1:
            logits = logits.view(-1)
            loss = self.criterion(logits, y)
        else:
            # y expected as class indices shape (B,)
            logits = logits.view(y.size(0), -1)
            loss = self.criterion(logits, y.long())

        opt.zero_grad(set_to_none=True)
        self.manual_backward(loss)
        opt.step()
        if sch is not None:
            sch.step()

        return loss, logits

    def training_step(self, batch, batch_idx):
        if self.num_classes == 1:
            y = batch["label"].float().view(-1)
        else:
            y = batch["label"].view(-1)

        loss1, _ = self._step_once(batch, y, mode="retriever")
        loss2, logits2 = self._step_once(batch, y, mode="predictor")

        loss = 0.5 * (loss1 + loss2)
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)

        if self.num_classes == 1:
            probs = torch.sigmoid(logits2.detach().view(-1))
            self.train_step_label.append(y.detach())
            self.train_step_preds.append(probs)

        return loss

    def on_train_epoch_end(self):
        if self.num_classes != 1:
            return
        y = torch.cat(self.train_step_label).long()
        p = torch.cat(self.train_step_preds)
        self.log("train_auroc", self.train_auroc(p, y), prog_bar=True)
        self.log("train_auprc", self.train_auprc(p, y), prog_bar=True)
        self.train_step_label.clear()
        self.train_step_preds.clear()

    def validation_step(self, batch, batch_idx):
        self.model.remed.set_mode("predictor")

        if self.num_classes == 1:
            y = batch["label"].float().view(-1)
            logits = self.forward(batch).view(-1)
            loss = self.criterion(logits, y)
            self.log("val_loss", loss, prog_bar=True, on_epoch=True)
            probs = torch.sigmoid(logits.detach())
            self.val_step_label.append(y.detach())
            self.val_step_preds.append(probs)
            return loss
        else:
            y = batch["label"].view(-1).long()
            logits = self.forward(batch).view(y.size(0), -1)
            loss = self.criterion(logits, y)
            self.log("val_loss", loss, prog_bar=True, on_epoch=True)
            return loss

    def on_validation_epoch_end(self):
        if self.num_classes != 1:
            return
        y = torch.cat(self.val_step_label).long()
        p = torch.cat(self.val_step_preds)
        self.log("val_auroc", self.val_auroc(p, y), prog_bar=True)
        self.log("val_auprc", self.val_auprc(p, y), prog_bar=True)
        self.val_step_label.clear()
        self.val_step_preds.clear()

    def test_step(self, batch, batch_idx):
        self.model.remed.set_mode("predictor")

        if self.num_classes == 1:
            y = batch["label"].float().view(-1)
            logits = self.forward(batch).view(-1)
            loss = self.criterion(logits, y)
            self.log("test_loss", loss, prog_bar=True, on_epoch=True)
            probs = torch.sigmoid(logits.detach())
            self.test_step_label.append(y.detach())
            self.test_step_preds.append(probs)
            return loss
        else:
            y = batch["label"].view(-1).long()
            logits = self.forward(batch).view(y.size(0), -1)
            loss = self.criterion(logits, y)
            self.log("test_loss", loss, prog_bar=True, on_epoch=True)
            return loss

    def on_test_epoch_end(self):
        if self.num_classes != 1:
            return
        y = torch.cat(self.test_step_label).long()
        p = torch.cat(self.test_step_preds)
        self.log("test_auroc", self.test_auroc(p, y))
        self.log("test_auprc", self.test_auprc(p, y))
        self.test_step_label.clear()
        self.test_step_preds.clear()

    def on_fit_start(self):
        m = self.model  # REMedWithGenHPF

        def count_trainable(module):
            return sum(p.numel() for p in module.parameters() if p.requires_grad)

        def count_total(module):
            return sum(p.numel() for p in module.parameters())

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

        if self.hparams.use_warmup:
            sch = LinearLR(
                opt, start_factor=1 / 100, end_factor=1.0, total_iters=self.hparams.warmup_steps
            )
        else:
            sch = LinearLR(opt, start_factor=1.0, end_factor=1.0, total_iters=1)

        return {"optimizer": opt, "lr_scheduler": sch}

In [60]:
from torch.utils.data import DataLoader

train_ds = REMedGenHPFPoolDataset(
    hf_path="./desc_gen_dataset/",
    data_idx_path="../downstream_idx.parquet",
    seq_field="within_stay_remed",                  
    time_field="within_stay_remed_time",
    time_diff_field="within_stay_remed_time_diff",
    label_field="y_mort_1yr",
    split="train",
    seq_len=511,
    max_tokens=64,
)


val_ds = REMedGenHPFPoolDataset(
    hf_path="./desc_gen_dataset/",
    data_idx_path="../downstream_idx.parquet",
    seq_field="within_stay_remed",                  
    time_field="within_stay_remed_time",
    time_diff_field="within_stay_remed_time_diff",
    label_field="y_mort_1yr",
    split="val",
    seq_len=511,
    max_tokens=64,
)

Loading dataset from disk:   0%|          | 0/306 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/306 [00:00<?, ?it/s]

In [61]:
# ===== imports =====
import math
import torch
import torch.nn as nn

# huggingface roformer
# from transformers import RoFormerConfig, RoFormerEncoder

# lightning + metrics
import lightning as lt
from torch.optim import Adam
from torch.optim.lr_scheduler import LinearLR
from torchmetrics.classification import BinaryAUROC, BinaryAveragePrecision


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 512):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, D)
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


def build_remed_experiment(
    *,
    vocab_size: int,
    pad_token_id: int = 0,
    # data
    max_tokens: int = 64,
    max_events: int = 511,          # must be >= your dataset S_max (e.g., seq_len)
    time_mode: str = "timestamp",   # "timestamp" or "time_diff"
    time_diff_unit: str = "days",   # "days" or "minutes"
    # model
    pred_dim: int = 128,
    pred_time: int = 48,
    max_retrieve_len: int = 128,
    n_layers: int = 2,
    n_heads: int = 8,
    dropout: float = 0.2,
    num_classes: int = 1,
    freeze_encoder: bool = True,
    # optim
    lr: float = 1e-5,
    max_epochs: int = 100,
    pos_weight: float = 1.0,
    use_warmup: bool = False,
    warmup_steps: int = 500,
):
    # ---- encoder ----
    encoder = GenHPFEncoder(
        vocab_size=train_ds.tokenizer.vocab_size,
        pad_token_id=train_ds.tokenizer.pad_token_id,
        encoder_embed_dim=128,
        encoder_layers=2,
        encoder_ffn_embed_dim=512,
        encoder_attention_heads=4,
        agg_embed_dim=128,      # IMPORTANT: must match pred_dim expected by REMed
        agg_layers=4,
        agg_ffn_embed_dim=512,
        agg_attention_heads=4,
        dropout=0.1,
        max_token_len=64,
        max_events=511,
        encoder_only=True,           # IMPORTANT: return per-event reprs (B,S,pred_dim)
        ckpt_path='/scratch/sas10092/ehr-foundation/models/simclr/wandb/run-20251224_094659-genhpf_simclr_genhpf_13612113_511_0_15_maskprob_12_5overlap/files/ckpt/epoch=62-step=21042.ckpt'
    )

    # ---- wrapper ----
    model = REMedWithGenHPF(
        genhpf_encoder=encoder,
        pred_dim=512, # check
        num_classes=1,
        pred_time=48,
        max_retrieve_len=128,
        n_heads=8,
        n_layers=2,
        dropout=0.2,
        freeze_encoder=True,
    )

    # ---- collator ----
    collator = REMedGenHPFCollator(
        pad_token_id=train_ds.tokenizer.pad_token_id,
#         time_mode=time_mode,
#         time_diff_unit=time_diff_unit,
    )

    # ---- lightning module ----
    lit = REMedLightningModule(
        model=model,
        lr=1e-5,
        max_epochs=75,
        pos_weight=1.0,
        freeze_encoder=True,
        use_warmup=True,
        warmup_steps=500,
        num_classes=1,
    )

    return {
        "encoder": encoder,
        "model": model,
        "collator": collator,
        "lit": lit,
    }


# ======================================================================
# EXAMPLE: your typical binary run
# ======================================================================
exp = build_remed_experiment(
    vocab_size=train_ds.tokenizer.vocab_size,
    pad_token_id=0,
    max_tokens=64,
    max_events=511,        # if your dataset seq_len is 512
    time_mode="timestamp",
    time_diff_unit="days",
    pred_dim=128,
    pred_time=48,
    max_retrieve_len=128,
    n_layers=2,
    n_heads=8,
    dropout=0.2,
    num_classes=1,
    freeze_encoder=True,
    lr=1e-5,
    max_epochs=10,
    pos_weight=1.0,
    use_warmup=False,
)
lit = exp["lit"]
collator = exp["collator"]

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collator, num_workers=8)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=True, collate_fn=collator, num_workers=8)
trainer = lt.Trainer(accelerator='auto', 
                    devices='auto',
                    strategy='auto',
#                             logger=wandb_logger, 
                    log_every_n_steps=1,
                    num_sanity_val_steps=0,
                    max_epochs=75,
                    precision='bf16-mixed', 
#                             callbacks=[early_stop,lr_monitor],
                    enable_checkpointing=False
                    )
trainer.fit(lit, train_dataloaders=train_loader,val_dataloaders=val_loader)

/home/sas10092/.conda/envs/med-ehr/lib/python3.9/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/sas10092/.conda/envs/med-ehr/lib/python3.9/sit ...
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


missing= 
[]
unexpected= 
['model.proj.0.weight', 'model.proj.0.bias', 'model.proj.2.weight', 'model.proj.2.bias']


/home/sas10092/.conda/envs/med-ehr/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type                   | Param

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [62]:
torch.cuda.get_device_capability()

(8, 0)

In [ ]:
# import numpy as np
# from datasets import load_from_disk
# from tqdm import tqdm

# ds = load_from_disk("./desc_gen_dataset")

# SEQ_FIELDS = [
#     "within24_descemb", "within48_descemb", "within_stay_descemb",
#     "within24_genhpf", "within48_genhpf", "within_stay_genhpf",
# ]

# def word_stats(ds, seq_field):
#     word_lens = []

#     for i in tqdm(range(3000), desc=seq_field):
#         events = ds[i][seq_field]
#         for e in events:
#             if e:
#                 word_lens.append(len(e.split()))

#     x = np.array(word_lens)
#     return {
#         "count": len(x),
#         "max": int(x.max()),
#         "p95": int(np.percentile(x, 95)),
#         "p99": int(np.percentile(x, 99)),
#         "mean": float(x.mean()),
#     }

# for f in SEQ_FIELDS:
#     stats = word_stats(ds, f)
#     print(f"\n{f}")
#     print(stats)